# 02 - Data Audit

Name: Usha Oudeacoumar
Student ID: 20057009
Email: 20057009@mydbs.ie

This notebook goes through every year of raw data (2011 to 2019) and checks what is
actually in there. For each year it counts how many rows there are, how many of our
planned feature columns actually exist that year, how many are missing values, and
what percent of patients were admitted. It saves these as csv tables in outputs/tables
so we know if the data is good enough to build a model, and what problems to watch
out for (like columns not existing in some years).

In [1]:
import sys
sys.path.append(".")
import os
import pandas as pd
from utils import STUDY_YEARS, OUT_TABLES, load_year_raw, add_admit_flag, filter_adults, load_json

os.makedirs(OUT_TABLES, exist_ok=True)

cfg = load_json("../configs/study_config.json")
feature_cols = cfg["feature_cols"]
feature_cols

['AGE',
 'SEX',
 'ETHIM',
 'RACER',
 'TEMPF',
 'PULSE',
 'RESPR',
 'BPSYS',
 'BPDIAS',
 'POPCT',
 'ARREMS',
 'IMMEDR',
 'PAINSCALE',
 'RESIDNCE',
 'RFV1']

In [2]:
# load every year, this can take a bit since it has to read the pdf codebooks too
yearly_data = {}
for yr in STUDY_YEARS:
    print("loading", yr)
    df = load_year_raw(yr)
    yearly_data[yr] = df
    print(yr, "rows:", len(df), "cols:", df.shape[1])

loading 2011
2011 rows: 31084 cols: 423
loading 2012
2012 rows: 29453 cols: 582
loading 2013
2013 rows: 24777 cols: 580
loading 2014
2014 rows: 23844 cols: 994
loading 2015
2015 rows: 21061 cols: 1013
loading 2016
2016 rows: 19467 cols: 943
loading 2017
2017 rows: 16709 cols: 943
loading 2018
2018 rows: 20291 cols: 944
loading 2019
2019 rows: 19481 cols: 905


In [3]:
# which feature columns actually exist every year (schema check)
rows = []
for yr in STUDY_YEARS:
    cols_here = set(yearly_data[yr].columns)
    row = {"yr": yr}
    for c in feature_cols:
        row[c] = c in cols_here
    rows.append(row)

col_check = pd.DataFrame(rows).set_index("yr")
col_check.to_csv(os.path.join(OUT_TABLES, "02_feature_columns_by_year.csv"))
col_check

,AGE,SEX,ETHIM,RACER,TEMPF,PULSE,RESPR,BPSYS,BPDIAS,POPCT,ARREMS,IMMEDR,PAINSCALE,RESIDNCE,RFV1
yr,,,,,,,,,,,,,,,
2011,True,True,True,True,True,True,True,True,True,True,True,True,False,True,True
2012,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2013,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2014,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2015,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2016,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2017,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2018,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2019,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


In [4]:
# missing value percent for each feature, per year, only adults, treating -9/-8/-7 codes as missing too
miss_rows = []
for yr in STUDY_YEARS:
    df = filter_adults(yearly_data[yr])
    n = len(df)
    for c in feature_cols:
        if c not in df.columns:
            miss_rows.append({"yr": yr, "col": c, "pct_missing": 100.0})
            continue
        num = pd.to_numeric(df[c], errors="coerce")
        bad = num.isin([-9, -8, -7]) | df[c].isna() | (df[c].astype(str).str.strip() == "")
        pct = 100 * bad.sum() / n
        miss_rows.append({"yr": yr, "col": c, "pct_missing": round(pct, 2)})

miss_df = pd.DataFrame(miss_rows)
miss_pivot = miss_df.pivot(index="col", columns="yr", values="pct_missing")
miss_pivot.to_csv(os.path.join(OUT_TABLES, "02_missingness_by_year.csv"))
miss_pivot

yr,2011,2012,2013,2014,2015,2016,2017,2018,2019
col,,,,,,,,,
AGE,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
ARREMS,5.49,3.71,4.93,2.67,2.96,2.78,2.78,3.44,3.04
BPDIAS,3.20,4.11,4.05,3.00,3.84,3.98,3.42,6.33,4.55
BPSYS,3.08,3.96,3.83,2.88,3.66,3.89,3.41,6.16,4.44
ETHIM,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
IMMEDR,0.00,19.80,20.38,22.31,22.25,21.34,20.99,23.92,22.97
PAINSCALE,100.00,22.74,27.77,26.80,26.05,28.78,30.03,33.44,32.90
POPCT,9.96,8.58,7.48,5.82,7.02,7.20,4.95,7.67,5.75
PULSE,5.19,6.15,5.40,5.40,8.24,7.89,5.17,8.88,6.53


In [5]:
# admit rate per year, adults only, this is our outcome prevalence check
summary_rows = []
for yr in STUDY_YEARS:
    df = filter_adults(yearly_data[yr])
    df = add_admit_flag(df)
    n = len(df)
    n_admit = df["admit"].sum()
    summary_rows.append({
        "yr": yr,
        "n_adult_rows": n,
        "n_admit": int(n_admit),
        "admit_rate": round(n_admit / n, 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT_TABLES, "02_admit_rate_by_year.csv"), index=False)
summary_df

,yr,n_adult_rows,n_admit,admit_rate
0,2011,24466,3648,0.1491
1,2012,23275,3059,0.1314
2,2013,19403,2327,0.1199
3,2014,18228,1983,0.1088
4,2015,16393,1724,0.1052
5,2016,15242,1556,0.1021
6,2017,12793,1519,0.1187
7,2018,15827,2177,0.1375
8,2019,14899,1896,0.1273
